# AIAP Past Year Practise Case Study 

In [ ]:
import sqlite3
import pandas as pd
import numpy as np


In [9]:
conn = sqlite3.connect("../data/score.db")

# sqlite_master is SQLite's built-in catalog: a hidden table that lists everything in the database (tables, indexes, etc.)
tables = pd.read_sql("SELECT name FROm sqlite_master WHERE type = 'table'", conn)
print(tables)

# Ttkes the first table's name and loads the entire table into a DataFrame
df = pd.read_sql(f"SELECT * FROM {tables.iloc[0,0]}", conn)
conn.close()

print(df.shape)
df.info()
print(df.isna().sum())
print(df.describe(include="all"))
df.head(5)

    name
0  score
(15900, 18)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15900 entries, 0 to 15899
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   index               15900 non-null  int64  
 1   number_of_siblings  15900 non-null  int64  
 2   direct_admission    15900 non-null  object 
 3   CCA                 15900 non-null  object 
 4   learning_style      15900 non-null  object 
 5   student_id          15900 non-null  object 
 6   gender              15900 non-null  object 
 7   tuition             15900 non-null  object 
 8   final_test          15405 non-null  float64
 9   n_male              15900 non-null  float64
 10  n_female            15900 non-null  float64
 11  age                 15900 non-null  float64
 12  hours_per_week      15900 non-null  float64
 13  attendance_rate     15122 non-null  float64
 14  sleep_time          15900 non-null  object 
 15  wake_time           159

,index,number_of_siblings,direct_admission,CCA,learning_style,student_id,gender,tuition,final_test,n_male,n_female,age,hours_per_week,attendance_rate,sleep_time,wake_time,mode_of_transport,bag_color
0,0,0,Yes,Sports,Visual,ACN2BE,Female,No,69.0,14.0,2.0,16.0,10.0,91.0,22:00,6:00,private transport,yellow
1,1,2,No,Sports,Auditory,FGXIIZ,Female,No,47.0,4.0,19.0,16.0,7.0,94.0,22:30,6:30,private transport,green
2,2,0,Yes,None,Visual,B9AI9F,Male,No,85.0,14.0,2.0,15.0,8.0,92.0,22:30,6:30,private transport,white
3,3,1,No,Clubs,Auditory,FEVM1T,Female,Yes,64.0,2.0,20.0,15.0,18.0,NaN,21:00,5:00,public transport,yellow
4,4,0,No,Sports,Auditory,AXZN2E,Male,No,66.0,24.0,3.0,16.0,7.0,95.0,21:30,5:30,public transport,yellow


## Data preprocessing

In [27]:
print(df['CCA'].unique())

print(df['learning_style'].unique())

print(df['student_id'].duplicated().sum())  # how many repeats
print(df.drop(columns=['index', 'bag_color']).duplicated().sum())
df[df['student_id'].duplicated(keep=False)].sort_values('student_id')   # show every row involved
cols = df.columns.drop(['index', 'bag_color'])
dup_rows = df[df['student_id'].duplicated(keep=False)]

# for each duplicated ID, count how many distinct values each column has
diff = dup_rows.groupby('student_id')[cols].nunique(dropna=False)

# keep only IDs where some column has more than 1 distinct value
conflicts = diff[(diff > 1).any(axis=1)]
print(len(conflicts))
print((conflicts > 1).sum())   # which columns cause the conflicts

print(df['age'].value_counts().sort_index())

['Sports' 'None' 'Clubs' 'Arts' 'ARTS' 'SPORTS' 'CLUBS' 'NONE']
['Visual' 'Auditory']
900
765
135
number_of_siblings     0
direct_admission       0
CCA                    0
learning_style         0
student_id             0
gender                 0
tuition                0
final_test            52
n_male                 0
n_female               0
age                    0
hours_per_week         0
attendance_rate       90
sleep_time             0
wake_time              0
mode_of_transport      0
dtype: int64
age
-5.0        4
-4.0        1
 5.0      216
 6.0      230
 15.0    7726
 16.0    7723
Name: count, dtype: int64


In [13]:
def data_processing(df):
    df = df.copy()

    # 1. Deduplicate: keep the most complete record per student
    df['n_missing'] = df.isna().sum(axis=1)
    df = (df.sort_values('n_missing')
            .drop_duplicates(subset='student_id', keep='first')
            .drop(columns='n_missing'))

    # 2. Standardise inconsistent categories
    df['CCA'] = df['CCA'].str.capitalize()                  # 'SPORTS' -> 'Sports'

    # 3. Remove impossible ages
    df = df[df['age'] > 0]                             

    # 4. Drop rows with no target (can't train or evaluate on them)
    df = df.dropna(subset=['final_test'])

    # 5. attendance_rate: NOT imputed here (see note below)

    return df.reset_index(drop=True)

,index,number_of_siblings,direct_admission,CCA,learning_style,student_id,gender,tuition,final_test,n_male,n_female,age,hours_per_week,attendance_rate,sleep_time,wake_time,mode_of_transport,bag_color
7932,7932,1,No,Sports,Auditory,UAMI3G,Female,No,52.0,3.0,26.0,-5.0,13.0,92.0,21:00,5:00,public transport,yellow
8602,8602,0,No,None,Visual,XQMSBU,Female,No,67.0,10.0,18.0,-5.0,18.0,91.0,22:00,6:00,private transport,green
8663,8663,0,Yes,Arts,Visual,39XWY2,Male,Yes,85.0,17.0,3.0,-5.0,5.0,94.0,23:00,7:00,walk,blue
8846,8846,2,No,Sports,Auditory,Z33FOS,Female,Yes,74.0,4.0,19.0,-5.0,13.0,90.0,22:00,6:00,private transport,white
